# Agentic WANDS BM25 + few-shot + judge

This notebook mirrors `configs/cheat-at-search/agentic_wands_bm25_fow_shot_judge.yml`.

ELI5: we teach a search agent to use a BM25 tool, give it a few labeled examples, and then have a separate judge agent tell it how good its results are. We also add simple validators to keep it searching until it has enough results and enough tool calls.

We'll build it step by step:
1. Introduce each validator (one at a time).
2. Introduce the tool guard.
3. Define the BM25 tool.
4. Add a function to inject few-shot examples.
5. Combine everything into a full agentic strategy.

In [ ]:
!pip install git+https://github.com/softwaredoug/cheat-at-search.git@ee2526eb8bfac087dc3f90522cc7191032e47dfd
from cheat_at_search.data_dir import mount
try:
    mount(use_gdrive=True)
except ImportError:
    from pathlib import Path
    manual_path = str(Path.home() / ".search-experiments" / "cheat-at-search")
    mount(use_gdrive=False, manual_path=manual_path)

## Get an OpenAI Key + load corpus

This will prompt you for an OpenAI Key to interact with GPT-5.

In [ ]:
import logging
import numpy as np
import pandas as pd

from openai import OpenAI
from cheat_at_search.data_dir import key_for_provider
from cheat_at_search.wands_data import corpus, judgments

OPENAI_KEY = key_for_provider("openai")
openai = OpenAI(api_key=OPENAI_KEY)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("agentic_wands_bm25_judge")

corpus = corpus.reset_index(drop=True)
doc_id_lookup = corpus["doc_id"].astype(str).to_numpy()
doc_id_to_index = {doc_id: idx for idx, doc_id in enumerate(doc_id_lookup)}

corpus[["doc_id", "title", "description"]].head(3)

## Sample queries (8-16)

We keep the notebook fast by sampling a small set of queries with a fixed seed.

In [ ]:
QUERY_COUNT = 12
queries = judgments[["query", "query_id"]].drop_duplicates()
queries = queries.sample(n=QUERY_COUNT, random_state=7).reset_index(drop=True)
queries

## Validator 1: require at least 10 results

ELI5: if the agent returns too few items, we ask it to keep searching.

In [ ]:
from pydantic import BaseModel, Field

class SearchResults(BaseModel):
    """The state of the search agent, which can be used to inform future reasoning and tool use."""
    ranked_results: list[str] = Field(description="Top ranked search results (their doc_ids) when complete")

def validate_num_results(resp: SearchResults | None, min_results: int = 10) -> str | None:
    if resp is None:
        return "Please return at least 10 results to give the user a good variety to choose from."
    ranked = resp.ranked_results or []
    if len(ranked) < min_results:
        return "Please return at least 10 results to give the user a good variety to choose from."
    return None

validate_num_results(SearchResults(ranked_results=["1", "2"]))

## Validator 2: require a minimum number of tool calls

ELI5: we want the agent to explore a little before it answers. So we make it call tools at least 4 times.

In [ ]:
def _tool_calls_from_inputs(inputs: list[dict]) -> int:
    count = 0
    for item in inputs:
        if isinstance(item, dict) and item.get("type") == "function_call_output":
            count += 1
    return count

def validate_tool_calls(inputs: list[dict], min_calls: int = 4) -> str | None:
    calls = _tool_calls_from_inputs(inputs)
    if calls < min_calls:
        return (
            "You're doing really well. Please keep searching until 4 tool calls have been made so no stone is left unturned."
        )
    return None

validate_tool_calls([], min_calls=4)

## Validator 3: LLM judge for relevance

ELI5: a second model looks at the results and gives emoji grades. If the grades are bad, we ask the agent to try again.

In [ ]:
from typing import Literal
from cheat_at_search.agent.openai_agent import OpenAIAgent

AllowedEmoji = Literal["😃", "😐", "😞"]

class GradedSearchResult(BaseModel):
    """A single judged search result with an emoji relevance label."""
    emoji: AllowedEmoji = Field(description="Emoji relevance label for this result.")
    title: str = Field(description="Document title for the judged result.")
    doc_id: str = Field(description="Document ID for the judged result.")

class LLMJudgeResponse(BaseModel):
    """Structured response from the LLM judge containing graded results."""
    graded_results: list[GradedSearchResult] = Field(default_factory=list)

JUDGE_PROMPT = """
You are a helpful assistant that judges the relevance of search results to a query.

Query: {query}

Results:
{results}

Please rate the relevance of these results to the query using emojis of how well they satisfy the query.

It's important that the item
* Is in the desired category / item type
* Preferable, but not required, they match the user's search terms

Respond as a list of graded results with fields: emoji, title, doc_id.
"""

def judge_relevance(query: str, results_text: str) -> LLMJudgeResponse:
    judge_agent = OpenAIAgent(
        tools=[],
        model="openai/gpt-5-mini",
        response_model=LLMJudgeResponse,
        reasoning_level="medium",
    )
    inputs = [
        {"role": "system", "content": "You judge relevance."},
        {"role": "user", "content": JUDGE_PROMPT.format(query=query, results=results_text)},
    ]
    return judge_agent.loop(inputs=inputs, agent_state={})

sample_results = "1 | Example Chair\n2 | Example Lamp"
judge_relevance("small chair", sample_results)

## Tool guard: disallow repeated queries

ELI5: if the agent tries the exact same query again, we block it and tell it to be more creative.

In [ ]:
def guard_disallow_repeated_queries(params: dict, agent_state: dict | None) -> str | None:
    """Reject repeated queries per tool using agent_state['past_queries']."""
    if agent_state is None:
        return None
    tool_name = params.get("tool_name")
    query = params.get("query")
    if not tool_name or query is None:
        return None
    q_guard = agent_state.setdefault("past_queries", {}).get(tool_name)
    if q_guard is None:
        q_guard = []
        agent_state["past_queries"][tool_name] = q_guard
    if query in q_guard:
        return (
            "Error! You've already tried query: "
            + query
            + " Be more creative and explore more!"
        )
    q_guard.append(query)
    agent_state["past_queries"][tool_name] = q_guard
    return None

guard_disallow_repeated_queries({"tool_name": "search_bm25_wands", "query": "chair"}, {"past_queries": {}})

## Define the BM25 tool (WANDS)

We build a BM25 search tool that matches the WANDS tool signature. The guard runs first.

In [ ]:
from typing import Union
from typing_extensions import Literal
from cheat_at_search.tokenizers import snowball_tokenizer

WANDS_TOP_CATEGORIES = [
    "Furniture",
    "Home Improvement",
    "Décor & Pillows",
    "Outdoor",
    "Storage & Organization",
    "Lighting",
    "Rugs",
    "Bed & Bath",
    "Kitchen & Tabletop",
    "Baby & Kids",
    "School Furniture and Supplies",
    "Appliances",
    "Holiday Décor",
    "Commercial Business Furniture",
    "Pet",
    "Contractor",
    "Sale",
    "Foodservice",
    "Shop Product Type",
    "Browse By Brand",
]
WandsProductCategory = Literal[
    "Furniture",
    "Home Improvement",
    "Décor & Pillows",
    "Outdoor",
    "Storage & Organization",
    "Lighting",
    "Rugs",
    "Bed & Bath",
    "Kitchen & Tabletop",
    "Baby & Kids",
    "School Furniture and Supplies",
    "Appliances",
    "Holiday Décor",
    "Commercial Business Furniture",
    "Pet",
    "Contractor",
    "Sale",
    "Foodservice",
    "Shop Product Type",
    "Browse By Brand",
]
WANDS_CATEGORY_COL = "category"

def _build_category_index(corpus_df, *, category_col: str):
    if category_col not in corpus_df.columns:
        raise ValueError(f"Missing {category_col} column for category filtering.")
    values = corpus_df[category_col].fillna("").astype(str)
    category_index = {}
    for idx, value in enumerate(values.tolist()):
        if not value:
            continue
        category_index.setdefault(value, []).append(idx)
    return {key: np.asarray(indices, dtype=int) for key, indices in category_index.items()}

def _category_indices(category_index, product_categories):
    if product_categories is None:
        return None
    if isinstance(product_categories, str):
        product_categories = [product_categories]
    indices = []
    for category in product_categories:
        indices.extend(category_index.get(category, []))
    if not indices:
        return np.asarray([], dtype=int)
    return np.asarray(sorted(set(indices)), dtype=int)

category_index = _build_category_index(corpus, category_col=WANDS_CATEGORY_COL)

def search_bm25_wands(
    keywords: str,
    product_categories: list[WandsProductCategory] | None = None,
    top_k: int = 5,
    agent_state=None,
) -> list[dict[str, Union[str, int, float]]]:
    """Search WANDS with BM25 over title/description and optional category filter.

    Args:
        keywords: The search query string.
        product_categories: Optional category filters. Categorization may be imperfect;
            consider searching both with and without categories.
        top_k: The number of top results to return (max 100).

    Returns:
        Search results as a list of dictionaries with 'id', 'title',
        'description', and 'score' keys.
    """
    guard_error = guard_disallow_repeated_queries(
        {"tool_name": "search_bm25_wands", "query": keywords},
        agent_state,
    )
    if isinstance(guard_error, str) and guard_error:
        return guard_error
    if top_k > 100:
        return "Error! top_k must be <= 100."

    indices = _category_indices(category_index, product_categories)
    if indices is None:
        working_corpus = corpus
    elif indices.size == 0:
        return []
    else:
        working_corpus = corpus.iloc[indices]

    bm25_scores = np.zeros(len(working_corpus))
    for term in snowball_tokenizer(keywords):
        bm25_scores += working_corpus["title_snowball"].array.score(term) * 10.0
        bm25_scores += working_corpus["description_snowball"].array.score(term) * 1.0

    top_k_indices = np.argsort(bm25_scores)[-top_k:][::-1]
    bm25_scores = bm25_scores[top_k_indices]
    top_rows = working_corpus.iloc[top_k_indices].copy()
    top_rows.loc[:, "score"] = bm25_scores

    results = []
    for _, row in top_rows.iterrows():
        result = {
            "id": row.get("doc_id", row.name),
            "title": row.get("title", ""),
            "description": row.get("description", ""),
            "score": row.get("score", 0.0),
        }
        if "path" in top_rows.columns:
            result["path"] = row.get("path", "")
        results.append(result)
    return results

search_bm25_wands("salon chair", top_k=3)

## Few-shot examples helper

ELI5: we show the agent a few examples of good/bad results so it learns what relevance looks like.

In [ ]:
import random

def _grade_column(judgments_df):
    for col in ("grade", "relevance", "rel", "label", "score"):
        if col in judgments_df.columns:
            return col
    return None

def _grade_to_emoji(grade, grade_levels):
    if not grade_levels:
        return "😐"
    if len(grade_levels) == 1:
        return "😐"
    if grade == grade_levels[0]:
        return "😭"
    if grade == grade_levels[-1]:
        return "😃"
    return "😐"

def _sorted_grades(values):
    def _coerce(value):
        try:
            return float(value)
        except (TypeError, ValueError):
            return None
    numeric = [value for value in values if _coerce(value) is not None]
    if len(numeric) == len(values):
        return sorted(numeric)
    return sorted(values, key=lambda value: str(value))

def build_few_shot_block(corpus_df, judgments_df, num_rows=6, seed=42):
    grade_col = _grade_column(judgments_df)
    if grade_col is None:
        raise ValueError("Judgments need a grade column.")
    pool = judgments_df.dropna(subset=[grade_col, "query", "doc_id"])
    grades = _sorted_grades(list(pool[grade_col].dropna().unique()))
    if not grades:
        return ""
    rng = random.Random(seed)
    grouped = {
        grade: pool[pool[grade_col] == grade].sample(
            frac=1.0, random_state=rng.randrange(1 << 30)
        )
        for grade in grades
    }
    queues = {grade: grouped[grade].iterrows() for grade in grades}
    samples = []
    while len(samples) < num_rows:
        advanced = False
        for grade in grades:
            try:
                _, row = next(queues[grade])
            except StopIteration:
                continue
            samples.append(row)
            advanced = True
            if len(samples) >= num_rows:
                break
        if not advanced:
            break

    corpus_lookup = None
    if "doc_id" in corpus_df.columns:
        corpus_lookup = corpus_df.set_index("doc_id", drop=False)

    lines = ["Few-shot examples (query, product, relevance):"]
    for row in samples:
        query = row.get("query")
        doc_id = row.get("doc_id")
        grade = row.get(grade_col)
        emoji = _grade_to_emoji(grade, grades)
        title = ""
        description = ""
        if corpus_lookup is not None and doc_id in corpus_lookup.index:
            match = corpus_lookup.loc[doc_id]
            if hasattr(match, "ndim") and match.ndim > 1:
                match = match.iloc[0]
            if hasattr(match, "get"):
                title = match.get("title", "")
                description = match.get("description", "")
        lines.extend(
            [
                f"Query: {query}",
                f"Doc ID: {doc_id}",
                f"Title: {title}",
                f"Description: {description}",
                f"Relevance: {emoji}",
                "",
            ]
        )
    return "\n".join(lines).strip()

few_shot_block = build_few_shot_block(corpus, judgments, num_rows=6, seed=42)
few_shot_block.splitlines()[:10]

## Full strategy: combine tools, guard, validators, and few-shot

We now wire everything together into an agentic strategy with a judge.

ELI5: the main agent searches with BM25, checks rules, asks the judge for feedback, and loops up to 2 times to improve.

In [ ]:
SYSTEM_PROMPT_BASE = """
You take user search queries and use search tools to find relevant products in a furniture / home goods catalog.

It's important to return results ranked from most to least relevant based on the user query.

You'll get feedback from a judge that evaluates relevance. It's criteria:
    * Is in the desired category / item type
    * Preferable, but not required, they match the user's search terms
So you'll go through many iterations of search. Pick the most relevant over all your searching and show
them at the very end.

You'll see some examples below of relevant / irrelevant items from the corpus.

Gather results until you have 10 best matches you can find. It's important to return at least 10.
Return the *DOC IDs*
"""

SYSTEM_PROMPT = SYSTEM_PROMPT_BASE
if few_shot_block:
    SYSTEM_PROMPT = SYSTEM_PROMPT.rstrip() + "\n\n" + few_shot_block + "\n"

class AgenticWandsBM25Judge(SearchStrategy):
    def __init__(self, corpus_df, model="gpt-5-mini", workers=1, max_loops=2):
        self.model = model
        self.max_loops = max_loops
        super().__init__(corpus_df, workers=workers)

    def _format_results_for_judge(self, doc_ids: list[str]) -> str:
        lines = []
        for doc_id in doc_ids:
            if str(doc_id) not in doc_id_to_index:
                continue
            row = corpus.iloc[doc_id_to_index[str(doc_id)]]
            title = row.get("title", "")
            lines.append(f"{doc_id} | {title}")
        return "\n".join(lines)

    def search(self, query: str, k: int = 10):
        inputs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Find me: {query}"},
        ]
        agent_state = {"past_queries": {}, "trace_logger": logger}
        agent = OpenAIAgent(
            tools=[search_bm25_wands],
            model=f"openai/{self.model}" if "/" not in self.model else self.model,
            response_model=SearchResults,
            reasoning_level="medium",
        )

        resp = None
        for _ in range(self.max_loops):
            resp, inputs, _ = agent.chat(inputs=inputs, agent_state=agent_state, logger=logger)

            num_results_prompt = validate_num_results(resp, min_results=10)
            if num_results_prompt:
                inputs.append({"role": "user", "content": num_results_prompt})
                continue

            tool_calls_prompt = validate_tool_calls(inputs, min_calls=4)
            if tool_calls_prompt:
                inputs.append({"role": "user", "content": tool_calls_prompt})
                continue

            ranked = resp.output_parsed.ranked_results if resp else []
            judge_text = self._format_results_for_judge(ranked)
            judge_response = judge_relevance(query, judge_text)
            if not judge_response.graded_results:
                inputs.append({"role": "user", "content": "Please return more relevant results to better help the user find what they're looking for."})
                continue

            worst = [r for r in judge_response.graded_results if r.emoji == "😞"]
            if worst:
                inputs.append({"role": "user", "content": "Please return more relevant results to better help the user find what they're looking for."})
                continue
            break

        ranked = resp.output_parsed.ranked_results if resp else []
        ranked = [str(doc_id) for doc_id in ranked]
        indices = [doc_id_to_index[doc_id] for doc_id in ranked if doc_id in doc_id_to_index][:k]
        return indices, [1.0] * len(indices)

strategy = AgenticWandsBM25Judge(corpus, workers=1)
strategy.search(queries.loc[0, "query"], k=5)

## Run a small benchmark

We run `run_strategy` on a small subset of queries and compute the mean NDCG.

In [ ]:
from cheat_at_search.search import run_strategy, ndcgs

results = run_strategy(strategy, judgments, num_queries=QUERY_COUNT, seed=7, cache=False)
ndcg_series = ndcgs(results)
pd.DataFrame({"metric": ["NDCG@10"], "value": [ndcg_series.mean()]})